# X-Ray Red Team Arena

Create a small, untargeted adversarial perturbation for one approved chest X-ray and submit it to your paired Blue Team. The protected evaluator checks the original source, measured perturbation, and model prediction.

**Threat model:** standard EfficientNet-B0, 256×256 grayscale X-ray represented as RGB, input values in [0, 1], and an overall measured L-infinity limit of 0.02. **Declared ε** is the attack-generator budget; **measured L∞** is calculated from the submitted PNG.

## Quick navigation

[1 · Choose a source](#red-setup) · [2 · Prepare a candidate](#red-prepare) · [3 · Submit an attack](#red-submit) · [4 · Review Blue’s response](#red-review)

The progress strip below highlights the current step. The turn banner tells you whether to act or wait.

<a id="red-setup"></a>
## 1. Confirm your assigned role and load approved sources

The Launcher-injected token determines your CTFd account, match, and role. Choose an approved `source_id`; do not upload a separate original image.

In [ ]:
identity = arena.identity()
display(identity)
assert identity['role'] == 'red', 'This workspace is not assigned to Red.'
display(HTML(arena.progress_header_html('red', 'setup')))
display(HTML(arena.turn_indicator_html()))
source_response = arena.get_sources()
sources = source_response['sources']
print(f"Approved sources: {len(sources)}; maximum L-infinity: {source_response['max_linf']}")
print('Choose an ID from the gallery in the source_id cell below.')
display(arena.source_gallery(sources))

In [ ]:
# Pick a thumbnail's visible ID; the first approved image is the default.
source_id = str(sources[0]['source_id'])  # Replace with an ID shown in the gallery.
source_info = next((item for item in sources if str(item['source_id']) == source_id), None)
if source_info is None:
    raise ValueError(f"Source ID {source_id!r} is not in the approved gallery.")
clean_path = arena.download_source(source_id)
clean_prediction = arena.predict_image(clean_path)
display(HTML(arena.source_info_card_html(source_info, clean_prediction)))
display(DisplayImage(filename=str(clean_path)))
display(HTML(arena.progress_header_html('red', 'prepare')))

<a id="red-prepare"></a>
## 2. Generate and inspect a candidate

The local checkpoint is provided for white-box experimentation. Notebook-side measurements help you inspect the candidate; the protected evaluator independently checks the submitted PNG. PGD is the default. After Blue responds, try `SPARSE_FGSM` with a smaller epsilon or change the approved source to adapt your next attack.

In [ ]:
model = arena.load_standard_model()
attack_method = 'PGD'  # also try 'FGSM' or 'SPARSE_FGSM'
attack_epsilon = 0.02
attack_step = 0.0025
attack_iterations = 8
candidate_path = arena.make_adversarial_candidate(
    source_id, method=attack_method, epsilon=attack_epsilon,
    step_size=attack_step, iterations=attack_iterations, sparse_fraction=0.01
)
print('Candidate:', candidate_path)
candidate_prediction = arena.predict_image(candidate_path)
display(HTML(arena.prediction_card_html(clean_prediction, candidate_prediction)))
display(DisplayImage(filename=str(candidate_path)))
display(HTML(arena.progress_header_html('red', 'prepare')))

In [ ]:
import numpy as np
from PIL import Image
clean = np.asarray(Image.open(clean_path).convert('RGB'), dtype=np.int16)
candidate = np.asarray(Image.open(candidate_path).convert('RGB'), dtype=np.int16)
measured_linf = float(np.abs(candidate-clean).max() / 255.0)
print(f'Declared ε: {attack_epsilon:.6f}')
print(f'Measured PNG L∞: {measured_linf:.6f} (exercise maximum: {source_response["max_linf"]:.6f})')
display(arena.compare_images(clean_path, candidate_path))

<a id="red-submit"></a>
## 3. Submit an attack

The protected evaluator checks the approved source, overall measured L-infinity limit, and misclassification before adding a submission to Blue’s inbox.

In [ ]:
submission = arena.submit_red_attack(
    candidate_path, source_id, attack_name=attack_method,
    epsilon=attack_epsilon, step_size=attack_step, iterations=attack_iterations
)
display(HTML(arena.attack_result_card_html(submission)))
display(Markdown(arena.round_summary_markdown()))
display(HTML(arena.progress_header_html('red', 'review')))
display(HTML(arena.turn_indicator_html()))

<a id="red-review"></a>
## 4. Review Blue’s response, then adapt

When Blue submits its defense configuration, review the labeled result card and match summary. Use Blue’s settings to choose a new approved source or attack strategy, then repeat the candidate and submission steps.

In [ ]:
response = arena.get_blue_response()
display(HTML(arena.blue_result_card_html(response)))
display(HTML(arena.turn_indicator_html()))
display(Markdown(arena.round_summary_markdown()))
display(HTML(arena.progress_header_html('red', 'review')))